# TF-IDF baselines

Trains the two TF-IDF + logistic-regression systems from [Legal Guidance in the Wild](https://huggingface.co/papers/2610.25159):

- a full-conversation cascade that detects guidance seeking and then assigns one of 14 primary topics;
- a prefix guidance model trained on cumulative user-turn prefixes.

Models are fit on the silver train split, selected on the silver validation split, and scored once on the 200-conversation gold set. Everything runs on CPU in a few minutes.

In [ ]:
%pip install -q "scikit-learn==1.9.0" "datasets>=3"

In [ ]:
import joblib
import numpy as np
import pandas as pd
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import FeatureUnion

SEED = 42
DEV_DATASET = "AmirMohseni/WildChat-Legal-Classification-V3-Hierarchical"
DEV_REVISION = "489b87c18e571e8c7e9a3b1c2463b79e50cef120"
GOLD_DATASET = "AmirMohseni/WildChat-Legal-Classification"
GOLD_REVISION = "65c37e93ba552b9d215bb2b3cde3135f5a75f1bc"
C_GRID = [0.25, 1.0, 4.0]
THRESHOLDS = np.linspace(0.05, 0.95, 91)
TOPICS = [
    "FAMILY_AND_ESTATES", "HOUSING_AND_PROPERTY", "EMPLOYMENT_AND_LABOR",
    "IMMIGRATION_AND_CITIZENSHIP", "CRIMINAL_LAW", "TAX_LAW",
    "CONSUMER_AND_PERSONAL_FINANCE", "BUSINESS_AND_COMMERCIAL",
    "TORTS_AND_CIVIL_LIABILITY", "GOVERNMENT_AND_ADMINISTRATIVE",
    "CIVIL_RIGHTS_AND_CONSTITUTIONAL", "DATA_PRIVACY_AND_TECHNOLOGY",
    "INTELLECTUAL_PROPERTY", "OTHER",
]


def user_turns(conversation):
    return [(i, m.get("content", "")) for i, m in enumerate(conversation) if m.get("role") == "user"]


def serialize(turns):
    return "\n".join(f"User: {content}" for _, content in turns)


def tfidf_features(char_max_df=1.0):
    return FeatureUnion([
        ("word", TfidfVectorizer(strip_accents="unicode", ngram_range=(1, 2), min_df=2, max_df=0.995,
                                 max_features=150_000, sublinear_tf=True, dtype=np.float32)),
        ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, max_df=char_max_df,
                                 max_features=200_000, sublinear_tf=True, dtype=np.float32)),
    ])


def macro_f1(y_true, y_pred, **kwargs):
    return f1_score(y_true, y_pred, average="macro", zero_division=0, **kwargs)


def positive_f1(y_true, y_pred, **kwargs):
    return f1_score(y_true, y_pred, zero_division=0, **kwargs)

In [ ]:
dev = load_dataset(DEV_DATASET, revision=DEV_REVISION)
train, val = dev["train"].to_pandas(), dev["validation"].to_pandas()
for frame in (train, val):
    frame["text"] = [serialize(user_turns(c)) for c in frame["conversation"]]
    frame["guidance"] = frame["seeks_legal_guidance"].astype(int)
print(f"train={len(train)} validation={len(val)}")

## Full-conversation cascade

Stage 1 picks `C` by validation macro-F1 and then the threshold by validation positive-class F1 (macro-F1 breaks ties). Stage 2 is trained only on guidance-positive conversations.

In [ ]:
guidance_features = tfidf_features().fit(train["text"])
Xg_train, Xg_val = guidance_features.transform(train["text"]), guidance_features.transform(val["text"])

guidance_candidates = {
    C: LogisticRegression(C=C, solver="liblinear", class_weight="balanced", max_iter=3000,
                          random_state=SEED).fit(Xg_train, train["guidance"])
    for C in C_GRID
}
guidance_C = max(C_GRID, key=lambda C: macro_f1(val["guidance"], guidance_candidates[C].predict(Xg_val)))
guidance_model = guidance_candidates[guidance_C]
val_prob = guidance_model.predict_proba(Xg_val)[:, 1]
guidance_threshold = max(
    THRESHOLDS,
    key=lambda t: (positive_f1(val["guidance"], val_prob >= t), macro_f1(val["guidance"], val_prob >= t)),
)
val_guidance = val_prob >= guidance_threshold
print(f"C={guidance_C} threshold={guidance_threshold:.2f} "
      f"validation macro-F1={macro_f1(val['guidance'], val_guidance):.4f}")

In [ ]:
topic_train, topic_val = train[train["guidance"] == 1], val[val["guidance"] == 1]
y_topic_train = topic_train["primary_topic"].map(TOPICS.index)
y_topic_val = topic_val["primary_topic"].map(TOPICS.index)

topic_features = tfidf_features().fit(topic_train["text"])
Xt_train, Xt_val = topic_features.transform(topic_train["text"]), topic_features.transform(topic_val["text"])
topic_candidates = {
    C: LogisticRegression(C=C, solver="lbfgs", class_weight="balanced", max_iter=3000,
                          random_state=SEED).fit(Xt_train, y_topic_train)
    for C in C_GRID
}
topic_C = max(C_GRID, key=lambda C: macro_f1(y_topic_val, topic_candidates[C].predict(Xt_val)))
topic_model = topic_candidates[topic_C]
print(f"C={topic_C} validation topic macro-F1={macro_f1(y_topic_val, topic_model.predict(Xt_val)):.4f}")


def predict_cascade(texts):
    prob = guidance_model.predict_proba(guidance_features.transform(texts))[:, 1]
    topic = [TOPICS[i] for i in topic_model.predict(topic_features.transform(texts))]
    return prob >= guidance_threshold, np.where(prob >= guidance_threshold, topic, "NO_GUIDANCE")


_, val_cascade = predict_cascade(val["text"])
val_target = np.where(val["guidance"] == 1, val["primary_topic"], "NO_GUIDANCE")
print(f"validation cascade accuracy={accuracy_score(val_target, val_cascade):.4f}")

## Prefix guidance model

One example per user turn: the prefix is positive once the silver guidance onset has been reached. Each prefix is weighted by `1 / number_of_user_turns`, so every conversation has total weight 1 in training and in model selection.

In [ ]:
def expand_prefixes(frame):
    rows = []
    for row in frame.itertuples():
        turns = user_turns(row.conversation)
        onset = row.first_guidance_user_turn_id if row.seeks_legal_guidance else None
        for n in range(1, len(turns) + 1):
            rows.append({
                "conversation_id": row.id,
                "text": serialize(turns[:n]),
                "label": int(onset is not None and turns[n - 1][0] >= onset),
                "weight": 1.0 / len(turns),
            })
    return pd.DataFrame(rows)


prefix_train, prefix_val = expand_prefixes(train), expand_prefixes(val)
print(f"train prefixes={len(prefix_train)} validation prefixes={len(prefix_val)}")

prefix_features = tfidf_features(char_max_df=0.995).fit(prefix_train["text"])
Xp_train, Xp_val = prefix_features.transform(prefix_train["text"]), prefix_features.transform(prefix_val["text"])
y_val, w_val = prefix_val["label"], prefix_val["weight"]


def weighted_scores(prob, threshold):
    pred = prob >= threshold
    return macro_f1(y_val, pred, sample_weight=w_val), positive_f1(y_val, pred, sample_weight=w_val)


prefix_candidates = []
for C in C_GRID:
    model = LogisticRegression(C=C, solver="liblinear", max_iter=2000, random_state=SEED)
    model.fit(Xp_train, prefix_train["label"], sample_weight=prefix_train["weight"])
    prob = model.predict_proba(Xp_val)[:, 1]
    threshold = max(THRESHOLDS, key=lambda t: (*weighted_scores(prob, t), -abs(t - 0.5)))
    prefix_candidates.append(((*weighted_scores(prob, threshold), -abs(C - 1.0)), C, model, threshold))

_, prefix_C, prefix_model, prefix_threshold = max(prefix_candidates, key=lambda c: c[0])
print(f"C={prefix_C} threshold={prefix_threshold:.2f} "
      f"weighted validation macro-F1={weighted_scores(prefix_model.predict_proba(Xp_val)[:, 1], prefix_threshold)[0]:.4f}")

## Gold evaluation

The gold labels are the adjudicated conversation-level decisions. Topic scores use the 84 guidance-positive conversations and average over the 12 topics they contain; end-to-end scores average over the outcomes observed in the gold set. Prefix models are scored after the first user turn, after the second user turn (only for the 104 conversations that have one), and after the final user turn.

In [ ]:
gold = load_dataset(GOLD_DATASET, "golden", split="test", revision=GOLD_REVISION).to_pandas()
gold["turns"] = [user_turns(c) for c in gold["conversation"]]
gold["text"] = gold["turns"].map(serialize)
y_gold = gold["seeks_legal_guidance"].astype(bool).to_numpy()
gold_target = np.where(y_gold, gold["primary_topic"], "NO_GUIDANCE")

gold_guidance, gold_cascade = predict_cascade(gold["text"])
gold_topic = np.array([TOPICS[i] for i in topic_model.predict(topic_features.transform(gold["text"][y_gold]))])
results = [
    ("Guidance", accuracy_score(y_gold, gold_guidance), macro_f1(y_gold, gold_guidance), .840, .839),
    ("Topic (gold positives)", accuracy_score(gold_target[y_gold], gold_topic),
     macro_f1(gold_target[y_gold], gold_topic, labels=sorted(set(gold_target[y_gold]))), .679, .559),
    ("End-to-end cascade", accuracy_score(gold_target, gold_cascade),
     macro_f1(gold_target, gold_cascade, labels=sorted(set(gold_target))), .740, .527),
]


def prefix_scores(n_turns, mask):
    texts = [serialize(turns[:n_turns or len(turns)]) for turns in gold["turns"][mask]]
    pred = prefix_model.predict_proba(prefix_features.transform(texts))[:, 1] >= prefix_threshold
    return accuracy_score(y_gold[mask], pred), macro_f1(y_gold[mask], pred)


has_two = gold["turns"].map(len).to_numpy() >= 2
everyone = np.ones(len(gold), dtype=bool)
results += [
    ("Prefix, turn 1", *prefix_scores(1, everyone), .830, .826),
    (f"Prefix, turn 2 (n={has_two.sum()})", *prefix_scores(2, has_two), .798, .798),
    ("Prefix, final turn", *prefix_scores(None, everyone), .825, .823),
]
pd.DataFrame(results, columns=["evaluation", "accuracy", "macro_f1", "paper_accuracy", "paper_macro_f1"]).round(3)

In [ ]:
artifacts = {
    "guidance_tfidf_logreg.joblib": {"features": guidance_features, "model": guidance_model,
                                     "threshold": guidance_threshold},
    "topic_tfidf_logreg.joblib": {"features": topic_features, "model": topic_model, "topic_names": TOPICS},
    "prefix_tfidf_logreg.joblib": {"features": prefix_features, "model": prefix_model,
                                   "threshold": prefix_threshold},
}
for path, artifact in artifacts.items():
    joblib.dump(artifact, path)